# Flux reconciliation for the ECOMICS experimental reaction fluxes

In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
import pandas as pd
from typing import Dict, List, Tuple, Optional
import warnings
import os
import sys
import cobra
from cobra.io import load_model

# Add paths
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../scripts'))

# Add directories
data_dir = "../data"

# Optimization Problem

**Data:**

- **MEAS** - indices of reactions with measured fluxes
- **INPUTS** - reactions with fixed fluxes (e.g. media components)
- $\bar{v}_j$ - measured reaction flux
- $S$ - stoichiometric matrix
- $d$ - vector of weights for each of the reaction rates in the inner LP (FBA)
- $v_L, v_U$ - lower / upper bounds on reaction rates

**Variables:**

- $v$ - reconciled reaction-flux vector
- $\lambda$ - dual multipliers for the mass-balance constraints
- $\alpha^L, \alpha^U$ - duals for lower/upper bounds



**Minimize:**
$$\min_{v, \lambda, \alpha^L, \alpha^U} \sum_{j \in MEAS} \left(\frac{v_j - \bar{v}_j}{\bar{v}_j}\right)^2$$

**Subject to:**

$$S v = 0 \quad \text{(steady-state mass balance)}$$

$$d + S_{int}^T \lambda + \alpha^L - \alpha^U = 0 \quad \text{(dual feasibility constraint)}$$

$$v_j = \bar{v}_j, \quad j \in \text{INPUTS} \quad \text{(fixed uptakes)}$$

$$v_L \leq v \leq v_U \quad \text{(flux bounds)}$$

$$(v_j - v_{L,j}) \alpha_j^L = 0, \quad (v_{U,j} - v_j) \alpha_j^U = 0, \quad j = 1, \ldots, n \quad \text{(complimentary constraints)}$$

$$\alpha^L \geq 0, \quad \alpha^U \geq 0 \quad \text{(non-negativity of dual variables)}$$

# Small network

In [4]:
# Network: Glucose_ext --(uptake)--> Glucose_int --(glycolysis)--> Pyruvate_int
#                                                                      |
#                                                         |------------|------------|
#                                                         v                         v
#                                                   Lactate_ext                Biomass

### 1. Model Inputs


In [5]:
# S matrix 

# Columns: reactions
#   1. Glucose_uptake
#   2. Pyr_formation
#   3. Lactate_formation
#   4. Biomass_formation

# Rows: metabolites 
#   1. Glucose_internal
#   2. Pyruvate_internal

S = np.array([
    [ 1, -1,  0,  0],  # Glucose_internal: +1 from uptake, -1 to pyruvate formation
    [ 0,  1, -1, -1],  # Pyruvate_internal: +1 from glucose, -1 to lactate, -1 to biomass  
])

metabolite_names = ['Glucose_internal', 'Pyruvate_internal']
reaction_names = ['Glucose_uptake', 'Pyr_formation', 'Lactate_formation', 'Biomass_formation']

n_metabolites = len(metabolite_names)
n_reactions = len(reaction_names)

# Reaction fluxes
# Note: all fluxes should be positive for forward reactions
# The S matrix handles the directionality (consumption vs production)
measurements = {
    'Glucose_uptake': -10,      # Positive: rate of glucose consumption
    'Pyr_formation': 10,       # Should equal glucose uptake due to mass balance
    'Lactate_formation': 9,    # Part of pyruvate goes to lactate
    'Biomass_formation': 7     # Part of pyruvate goes to biomass
}

# Fixed fluxes
fixed_fluxes = {
    'Glucose_uptake': 10.0  # Fix glucose uptake rate
}

# Flux bounds - all reactions are forward (positive)
flux_bounds = {
    'Glucose_uptake': (0, 20),    # Forward only (glucose consumption)
    'Pyr_formation': (0, 20),     # Forward only
    'Lactate_formation': (0, 15), # Forward only
    'Biomass_formation': (0, 10)  # Forward only
}

### 2. Check mass balance consistency

Mass balance constraints from S matrix:
1. Glucose_internal: v_glucose_uptake - v_pyr_formation = 0
   => v_glucose_uptake = v_pyr_formation
2. Pyruvate_internal: v_pyr_formation - v_lactate_formation - v_biomass_formation = 0
   => v_pyr_formation = v_lactate_formation + v_biomass_formation

In [6]:
# Check mass balance consistency of measurements
print("Current measurements:")
for reaction, flux in measurements.items():
    print(f"  {reaction}: {flux}")

# Check constraints
glucose_uptake = measurements['Glucose_uptake']
pyr_formation = measurements['Pyr_formation'] 
lactate_formation = measurements['Lactate_formation']
biomass_formation = measurements['Biomass_formation']

print(f"\nConstraint 1 check:")
print(f"  Glucose uptake: {glucose_uptake}")
print(f"  Pyruvate formation: {pyr_formation}")
print(f"  Difference: {abs(glucose_uptake - pyr_formation)}")

print(f"\nConstraint 2 check:")
print(f"  Pyruvate formation: {pyr_formation}")
print(f"  Lactate + Biomass: {lactate_formation + biomass_formation}")
print(f"  Difference: {abs(pyr_formation - (lactate_formation + biomass_formation))}")


Current measurements:
  Glucose_uptake: -10
  Pyr_formation: 10
  Lactate_formation: 9
  Biomass_formation: 7

Constraint 1 check:
  Glucose uptake: -10
  Pyruvate formation: 10
  Difference: 20

Constraint 2 check:
  Pyruvate formation: 10
  Lactate + Biomass: 16
  Difference: 6


### 3. Set the fluxes bounds

In [7]:
# Set flux bounds
bounds = {}

# Default bounds
default_lower = -1000.0
default_upper = 1000.0

for i, reaction_name in enumerate(reaction_names):
    if flux_bounds and reaction_name in flux_bounds:
        bounds[i] = flux_bounds[reaction_name]
    else:
        bounds[i] = (default_lower, default_upper)

### 4. Setting up the flux reconciliation model

In [8]:
# Initialize Pyomo model
model = pyo.ConcreteModel()
        
# Sets
model.metabolites = pyo.Set(initialize=range(n_metabolites))
model.reactions = pyo.Set(initialize=range(n_reactions))
model.measured_reactions = pyo.Set(initialize=[
    j for j, name in enumerate(reaction_names)
    if name in measurements 
])

# Variables
# Reaction fluxes
model.v = pyo.Var(model.reactions, bounds=(-1000, 1000))

# Dual variables for optimiality conditions
model.alpha_L = pyo.Var(model.reactions, bounds=(0, 1000))    # Lower bound multipliers
model.alpha_U = pyo.Var(model.reactions, bounds=(0, 1000))    # Upper bound multipliers

# Apply flux bounds to reaction variables
for i in model.reactions:
    model.v[i].setlb(bounds[i][0])
    model.v[i].setub(bounds[i][1])
    
# Fixed fluxes 
if fixed_fluxes:
    for reaction_name, flux_value in fixed_fluxes.items():
        if reaction_name in reaction_names:
            reaction_idx = reaction_names.index(reaction_name)
            model.v[reaction_idx].fix(flux_value)

In [9]:
# Vector of initial weights (zeros). Each entry corresponds to a reaction
objective_weights = np.zeros(n_reactions)

# Assign weight of 1.0 to reaction of objective function
objective_reaction = 'Biomass_formation'
reaction_idx = reaction_names.index(objective_reaction)
objective_weights[reaction_idx] = 1.0

In [10]:
# Objective function: minimizing the deviation of estimated rates from measured rates
def objective_rule(m):
    return sum(
        ((m.v[j] - measurements[reaction_names[j]])/measurements[reaction_names[j]])**2
        for j in m.measured_reactions
    )
model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

# Mass balance constraints (S matrix)
def mass_balance_rule(m, i):
    return sum(S[i, j] * m.v[j] for j in m.reactions) == 0
model.mass_balance = pyo.Constraint(model.metabolites, rule=mass_balance_rule)


#### **NOTE:** Uncomment the next cell to include optimality constraints 

In [11]:
# Parameters
barrier_param = 1e-6

# Dual feasibility constraint
def stationarity_rule(m, j):
    return (objective_weights[j] + m.alpha_L[j] - m.alpha_U[j] == 0)
model.stationarity = pyo.Constraint(model.reactions, rule=stationarity_rule)
    
# Complementarity constraints
def comp_lower_rule(m, j):
    return (m.v[j] - bounds[j][0]) * m.alpha_L[j] <= barrier_param
model.comp_lower = pyo.Constraint(model.reactions, rule=comp_lower_rule)
        
def comp_upper_rule(m, j):
    return (bounds[j][1] - m.v[j]) * m.alpha_U[j] <= barrier_param
model.comp_upper = pyo.Constraint(model.reactions, rule=comp_upper_rule)


### 5. Solving the model

In [12]:
# Configure solver
solver_options = {
    'max_iter': 3000,
    'tol': 1e-8,
    'acceptable_tol': 1e-6,
}

solver = pyo.SolverFactory('ipopt')

for key, value in solver_options.items():
        solver.options[key] = value

# Solve
try:
    results = solver.solve(model, tee=True)
    
    if (results.solver.termination_condition == TerminationCondition.optimal or
        results.solver.termination_condition == TerminationCondition.locallyOptimal):
        
        # Extract results
        reconciled_fluxes = {
            reaction_names[j]: pyo.value(model.v[j])
            for j in model.reactions
        }
        
        objective_value = pyo.value(model.obj)
        
        results_dict = {
            'reconciled_reaction_fluxes': reconciled_fluxes,
            'objective_value': objective_value,
            'solver_status': 'optimal',
            'original_measurements': measurements,
        }
        
        print(f"Objective value: {objective_value}")
        
    else:
        print(f"Solver terminated with condition: {results.solver.termination_condition}")
        results_dict = {'solver_status': 'failed', 'termination_condition': str(results.solver.termination_condition)}
        
except Exception as e:
    print(f"Error during solving: {str(e)}")
    results_dict = {'solver_status': 'error', 'error_message': str(e)}

Ipopt 3.14.17: max_iter=3000
tol=1e-08
acceptable_tol=1e-06


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.17, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:       12
Number of nonzeros in inequality constraint Jacobian.:       14
Number of nonzeros in Lagrangian Hessian.............:        9

Total number of variables............................:       11
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       11
                     variables with only upper bounds:        0
Total number of equality constr

### TAKEAWAYS
When using the Optimality constraints the optimization results change significantly. The objective value is also much higher. Could this mean that the optimality constraints lead to worse results? (for this small network they do)

- Obj value without optimality constraints: 4.28
- Obj value with optimality constraints: 5.18

# ECOMICS fluxomics data

### 1. Prep the fluxomics data

In [13]:
# Load data
condition_string = 'W3110_MD121_M9+Glu_none'
exp_fluxes_path = os.path.join(data_dir, "processed", "ECOMICS", f"fluxomics_{condition_string}.csv")
exp_fluxes_df = pd.read_csv(exp_fluxes_path)

# Rename the rxn_id column
exp_fluxes_df.rename(columns={'exp_reaction': 'rxn_id'}, inplace=True)

# Remove the 'R000X_' prefix
exp_fluxes_df['rxn_id'] = exp_fluxes_df['rxn_id'].str.replace(r'^R\d+_', '', regex=True)

In [14]:
# Group by rxn_id and for multiple matches, take the first one (190 --> 145)
# TO DO: These should be manually checked with the output of ECOMICS_fluxomics.ipynb
exp_fluxes_df = exp_fluxes_df.groupby('rxn_id', as_index=False).agg({'exp_flux': 'first'})

In [15]:
# Divide by 10 because the glucose uptake rate will be set to 10 mmol/gDW/h
exp_fluxes_df.iloc[:, 1] = exp_fluxes_df.iloc[:, 1] / 10

In [16]:
# Remove 0 fluxes --> these are actually NaN values
exp_fluxes_df = exp_fluxes_df[exp_fluxes_df['exp_flux'] != 0]

### 2. Set the fixed fluxes (e.g. medium components)
TO DOs:
- Check if 'fixing' to an exact number works better than setting the upper/lower bounds
- Check if discarding some of these (e.g. co2, o2) would give better results

In [17]:
condition_string = 'W3110_MD121_M9+Glu_none'
fixed_fluxes_path = os.path.join(data_dir, "processed", "ECOMICS", f"medium_{condition_string}.csv")
fixed_fluxes_df = pd.read_csv(fixed_fluxes_path)

In [18]:
# Inspect the fixed fluxes dataframe
print(fixed_fluxes_df)

     rxn_id  exp_fixed_flux
0  EX_co2_e           175.2
1   EX_o2_e           158.6
2  EX_nh4_e            50.1
3  EX_so4_e             1.7


In [19]:
# Add a row for glucose uptake
new_row = pd.DataFrame([{'rxn_id': 'EX_glc__D_e', 'exp_fixed_flux': 100}])
fixed_fluxes_df = pd.concat([fixed_fluxes_df, new_row], ignore_index=True)

# Divide by 10 because the glucose uptake rate will be set to 10 mmol/gDW/h
fixed_fluxes_df.iloc[:, 1] = fixed_fluxes_df.iloc[:, 1] / 10

# Convert to negative values (they are uptakes)
fixed_fluxes_df.iloc[:, 1] = fixed_fluxes_df.iloc[:, 1] * -1

print(fixed_fluxes_df)

        rxn_id  exp_fixed_flux
0     EX_co2_e          -17.52
1      EX_o2_e          -15.86
2     EX_nh4_e           -5.01
3     EX_so4_e           -0.17
4  EX_glc__D_e          -10.00


In [20]:
# # Export to csv
# fixed_fluxes_df.to_csv(os.path.join(data_dir, f"fixed_fluxes_df.csv"), index=False)

In [21]:
# Minimal fixed fluxes using just EX_glc__D_e
minimal_fixed_fluxes = fixed_fluxes_df[fixed_fluxes_df['rxn_id'] == 'EX_glc__D_e'].copy()
print(minimal_fixed_fluxes)

        rxn_id  exp_fixed_flux
4  EX_glc__D_e           -10.0


# *E. coli* core model

In [22]:
# Load model
core_model_path = os.path.join(data_dir, "raw", "GEMs", "e_coli_core.xml")
core_model = cobra.io.read_sbml_model(core_model_path)

print(f'Number of reactions for core model: {len(core_model.reactions)}')

Number of reactions for core model: 95


In [23]:
# FBA check
core_model.optimize()
print(core_model.objective.value)

0.8739215069684302


In [24]:
for key, value in core_model.medium.items():
    print(f"{key}: {value}")

EX_co2_e: 1000.0
EX_glc__D_e: 10.0
EX_h_e: 1000.0
EX_h2o_e: 1000.0
EX_nh4_e: 1000.0
EX_o2_e: 1000.0
EX_pi_e: 1000.0


#### Using minimal fixed fluxes - just glucose uptake set to -10 $\frac{1}{h}$

In [42]:
from scripts.flux_reconciliation_module import flux_reconciliation

recon_minimal_core_results = flux_reconciliation(core_model_path, 
                              exp_fluxes_df, 
                              minimal_fixed_fluxes,
                              obj_rxn='BIOMASS_Ecoli_core_w_GAM',
                              solver_name='ipopt'
                              )

Model loaded successfully from ../data\raw\GEMs\e_coli_core.xml
Model contains 95 reactions and 72 metabolites

=== Model Information ===
Number of reactions: 95
Number of metabolites: 72
S matrix shape: (72, 95)
Number of non-zero elements in S: 360
Sample of flux bounds:
PFK: (0.0, 1000.0)
PFL: (0.0, 1000.0)
PGI: (-1000.0, 1000.0)
PGK: (-1000.0, 1000.0)
PGL: (0.0, 1000.0)

=== Validating Stoichiometric Matrix ===

=== Measured Fluxes Information ===
Total measured reactions: 128
Matched measurements: 33
Missing reactions: 95
First few missing reactions: ['ACHBS', 'ACLS', 'ACt2rpp', 'ACtex', 'ADSK']

Sample of measurements:
ACKr: flux = 6.590000000000001
ACONTa: flux = 1.72
ACONTb: flux = 1.72
AKGDH: flux = 1.3800000000000001
CS: flux = 2.5

=== Fixed Fluxes Information ===
Total fixed reactions: 1
Matched fixed fluxes: 1
Missing reactions: 0

All fixed fluxes:
EX_glc__D_e: -10.0

=== Validating Measurements Against Bounds ===

=== Optimization Model Setup ===
Sample of bounds:
PFK: (

c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACHBS not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACLS not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACt2rpp not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACtex not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - Unive

#### Using all fixed fluxes

In [43]:
from scripts.flux_reconciliation_module import flux_reconciliation

recon_core_results = flux_reconciliation(core_model_path, 
                              exp_fluxes_df, 
                              fixed_fluxes_df,
                              obj_rxn='BIOMASS_Ecoli_core_w_GAM',
                              solver_name='ipopt'
                              )

Model loaded successfully from ../data\raw\GEMs\e_coli_core.xml
Model contains 95 reactions and 72 metabolites

=== Model Information ===
Number of reactions: 95
Number of metabolites: 72
S matrix shape: (72, 95)
Number of non-zero elements in S: 360
Sample of flux bounds:
PFK: (0.0, 1000.0)
PFL: (0.0, 1000.0)
PGI: (-1000.0, 1000.0)
PGK: (-1000.0, 1000.0)
PGL: (0.0, 1000.0)

=== Validating Stoichiometric Matrix ===

=== Measured Fluxes Information ===
Total measured reactions: 128
Matched measurements: 33
Missing reactions: 95
First few missing reactions: ['ACHBS', 'ACLS', 'ACt2rpp', 'ACtex', 'ADSK']

Sample of measurements:
ACKr: flux = 6.590000000000001
ACONTa: flux = 1.72
ACONTb: flux = 1.72
AKGDH: flux = 1.3800000000000001
CS: flux = 2.5

  Measured value: 5.01
  Fixed value: -5.01
Using fixed value and not measured value.

=== Fixed Fluxes Information ===
Total fixed reactions: 5
Matched fixed fluxes: 4
Missing reactions: 1
Missing reactions: ['EX_so4_e']

Conflicting reactions:
E

c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACHBS not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACLS not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACt2rpp not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - University of Toronto\GitHub\CatNivoDataset\scripts\flux_reconciliation_module.py:113: UserWarning: Reaction ACtex not found in model reactions
  warnings.warn(f"Reaction {rxn_id} not found in model reactions")
c:\Users\lyachinas\OneDrive - Unive

#### TAKEAWAY: Using minimal fluxes vs all fixed fluxes is the same (both result in 26.15)
#### What really makes a change is **fixing** the fluxes. This makes the second optimization using the whole set of fixed fluxes infeasible. It also gives a worse objective function result (over 560 for the minimization).
#### The problem now sets the upper/lower bounds using the fixed flux values
- Minimal obj result = 26.15
- All fixed obj result = 26.15

# iML1515 Model

#### Using minimal fixed fluxes

In [44]:
from scripts.flux_reconciliation_module import flux_reconciliation

iML1515_path = os.path.join(data_dir, "raw", "GEMs", "iML1515_GEM.xml")

recon_minimal_iml1515_results = flux_reconciliation(iML1515_path,
                              exp_fluxes_df,
                              minimal_fixed_fluxes,
                              obj_rxn='BIOMASS_Ec_iML1515_core_75p37M',
                              solver_name='ipopt'
                              )

Model loaded successfully from ../data\raw\GEMs\iML1515_GEM.xml
Model contains 2712 reactions and 1877 metabolites

=== Model Information ===
Number of reactions: 2712
Number of metabolites: 1877
S matrix shape: (1877, 2712)
Number of non-zero elements in S: 10575
Sample of flux bounds:
ALATA_D2: (0.0, 1000.0)
SHCHD2: (0.0, 1000.0)
CPPPGO: (0.0, 1000.0)
GTHOr: (-1000.0, 1000.0)
DHORD5: (0.0, 1000.0)

=== Validating Stoichiometric Matrix ===

=== Measured Fluxes Information ===
Total measured reactions: 128
Matched measurements: 128
Missing reactions: 0

Sample of measurements:
ACHBS: flux = 0.2
ACKr: flux = 6.590000000000001
ACLS: flux = 0.29
ACONTa: flux = 1.72
ACONTb: flux = 1.72

=== Fixed Fluxes Information ===
Total fixed reactions: 1
Matched fixed fluxes: 1
Missing reactions: 0

All fixed fluxes:
EX_glc__D_e: -10.0

=== Validating Measurements Against Bounds ===

=== Optimization Model Setup ===
Sample of bounds:
ALATA_D2: (0.0, 1000.0)
SHCHD2: (0.0, 1000.0)
CPPPGO: (0.0, 1000.0)

#### Using all fixed fluxes

In [47]:
from scripts.flux_reconciliation_module import flux_reconciliation

iML1515_path = os.path.join(data_dir, "raw", "GEMs", "iML1515_GEM.xml")

recon_iml1515_results = flux_reconciliation(iML1515_path,
                              exp_fluxes_df,
                              fixed_fluxes_df,
                              obj_rxn='BIOMASS_Ec_iML1515_WT_75p37M',
                              )

Model loaded successfully from ../data\raw\GEMs\iML1515_GEM.xml
Model contains 2712 reactions and 1877 metabolites

=== Model Information ===
Number of reactions: 2712
Number of metabolites: 1877
S matrix shape: (1877, 2712)
Number of non-zero elements in S: 10575
Sample of flux bounds:
ALATA_D2: (0.0, 1000.0)
SHCHD2: (0.0, 1000.0)
CPPPGO: (0.0, 1000.0)
GTHOr: (-1000.0, 1000.0)
DHORD5: (0.0, 1000.0)

=== Validating Stoichiometric Matrix ===

=== Measured Fluxes Information ===
Total measured reactions: 128
Matched measurements: 128
Missing reactions: 0

Sample of measurements:
ACHBS: flux = 0.2
ACKr: flux = 6.590000000000001
ACLS: flux = 0.29
ACONTa: flux = 1.72
ACONTb: flux = 1.72

  Measured value: 5.01
  Fixed value: -5.01
Using fixed value and not measured value.

  Measured value: 0.16999999999999998
  Fixed value: -0.16999999999999998
Using fixed value and not measured value.

=== Fixed Fluxes Information ===
Total fixed reactions: 5
Matched fixed fluxes: 5
Missing reactions: 0

